# Sanity Check
## 03 - Detect Noisy Channels

In [1]:
def visualize_noisy_channels(subj, person="P1"):
    """
    Load the noisy-channel-cleaned raw file and visualize:
    1) EEG traces
    2) Marked bad channels
    3) Topomap of channel variance (works with MNE 1.8 + BioSemi)
    """
    # File from Step 03
    file_path = OUTPUT_DIR / f"sub-{subj}_{person}_raw_noisy_cleaned.fif"
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return

    # Load raw data
    raw = load_raw(OUTPUT_DIR / f"sub-{subj}_{person}_raw.fif", preload=True)

    # Ensure montage exists (BioSemi ActiveTwo 64)
    try:
        raw.get_montage()
        print("Montage found in raw data.")
    except RuntimeError:
        print("Montage not available.")

    # Print info
    print(raw)
    print("Bad channels marked in raw.info['bads']:", raw.info['bads'])

    # --- Plot EEG channels with bad channels highlighted ---
    raw.plot(n_channels=32, block=True,
             title=f"Subject {subj} {person} - Bad Channels Highlighted")

    # ----------------------------------------------------------------------
    # ---------------------- Topomap of Channel Variance -------------------
    # ----------------------------------------------------------------------
    import numpy as np

    picks = mne.pick_types(raw.info, eeg=True)
    data = raw.get_data(picks=picks)
    ch_names = [raw.ch_names[i] for i in picks]

    variances = data.var(axis=1)

    # Get BioSemi layout positions (works without digitizer points)
    layout = mne.find_layout(raw.info)
    pos = layout.pos  # 2D positions of electrodes

    # Create topomap
    im, _ = mne.viz.plot_topomap(
        variances,
        pos,
        cmap="Reds",
        contours=0,
        show=False,
        sensors=False  # we label manually
    )

    # Add channel name labels manually
    for name, (x, y) in zip(ch_names, pos):
        plt.text(x, y, name, fontsize=6, ha='center', va='center')

    plt.colorbar(im)
    plt.title(f"Channel variance - Subject {subj} {person}")
    plt.show()